# Kohn-Sham Operator Inspection

This notebook looks inside the current Γ-point toy DFT Hamiltonian. The purpose is to see the separate operator pieces that act on an orbital: kinetic `T`, local pseudopotential `V_local`, Hartree `V_H[ρ]`, and exchange-correlation `V_xc[ρ]`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.dft import DFTSystem, KohnShamOperator, SCFConfig, run_scf

We use a tiny one-center Gaussian well. This is not chemically reliable DFT; it is a controlled numerical target where each term is easy to inspect.

In [ ]:
system = DFTSystem.one_center(
    cell=(6.0, 6.0, 6.0),
    grid_shape=(4, 4, 4),
    center=(3.0, 3.0, 3.0),
    amplitude=-2.5,
    width=0.8,
)
config = SCFConfig(max_iterations=2, solver="dense", seed=11)
result = run_scf(system, config=config)
operator = KohnShamOperator.from_density(
    system.grid,
    system.pseudopotential.field(system.grid),
    result.density,
)
result.to_dict()

The same orbital is passed through each operator component. The norms below give a compact scale comparison; the final residual says how close the SCF orbital is to an eigenvector of the current `H_KS[ρ]`.

In [ ]:
psi = result.orbitals[0]
components = {
    "Tψ": operator.apply_kinetic(psi),
    "V_localψ": operator.apply_local_potential(psi),
    "(V_H+V_xc)ψ": operator.apply_hartree_xc_potential(psi),
    "Hψ": operator.apply_hamiltonian(psi),
}
{name: float(np.linalg.norm(np.array(value).ravel())) for name, value in components.items()}

In [ ]:
mid = system.grid.shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
images = [
    (result.density, "ρ(r)"),
    (operator.local_potential, "V_local"),
    (operator.effective_potential, "V_eff"),
]
for ax, (field, title) in zip(axes, images, strict=True):
    image = ax.imshow(np.array(field)[:, :, mid], origin="lower")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
fig.suptitle("central z-slice")